# Entrega 2 — Data Science
## Painel Ligue 180 · Análise Exploratória Completa
**Projeto Integrador | Violência contra a Mulher — Aplicativo de Apoio Camuflado**  
**2º Semestre de 2026**

> ⚠️ **Aviso de protótipo acadêmico:** Os dados usados são públicos e agregados (não há dados pessoais de vítimas). Este notebook foi produzido exclusivamente para fins acadêmicos.

---

### Divisão de etapas e autoria

| Seção | Etapa | Responsável |
|-------|-------|-------------|
| 1 | Escolha e governança da base | _Preencher nome_ |
| 2 | Diagnóstico de qualidade | _Preencher nome_ |
| 3 | Tratamento dos dados | _Preencher nome_ |
| 4 | Análise exploratória | _Preencher nome_ |
| 5 | Plano de modelagem | _Preencher nome_ |

---
## Seção 0 — Configuração e Importações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# --- Estilo visual ---
plt.rcParams.update({
    'figure.facecolor': '#07111f',
    'axes.facecolor': '#111e31',
    'axes.edgecolor': '#263750',
    'axes.labelcolor': '#a5b4cc',
    'xtick.color': '#6b7fa0',
    'ytick.color': '#6b7fa0',
    'text.color': '#f4f7ff',
    'grid.color': '#1d2d44',
    'grid.linestyle': '--',
    'grid.alpha': 0.6,
    'font.family': 'DejaVu Sans',
    'figure.titlesize': 14,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
COR_PRINCIPAL = '#79e3d1'
COR_DESTAQUE  = '#ff9c9c'
COR_SECUNDARIA = '#5b9cf6'

print('Bibliotecas carregadas com sucesso.')

---
## Seção 1 — Escolha e Governança da Base
_Responsável: [preencher nome]_

### 1.1 Ficha da Base

| Campo | Descrição |
|-------|-----------|
| **Nome** | Painel Ligue 180 — Central de Atendimento à Mulher |
| **Órgão responsável** | Ministério das Mulheres / Secretaria de Políticas para as Mulheres (SPM) |
| **Link da fonte** | https://www.gov.br/mulheres/pt-br/assuntos/ouvidoria-da-mulher/ligue-180 |
| **Data de acesso** | Outubro de 2026 |
| **Licença** | Dados públicos — Lei de Acesso à Informação (LAI 12.527/2011) |
| **Período coberto** | 2014–2023 |
| **Granularidade** | Mensal, por UF e categoria de violência |
| **Formato** | CSV (painéis de dados abertos) |

### 1.2 Justificativa

O **Ligue 180** é a central nacional de atendimento e denúncias de violência contra a mulher.
Seus dados agregados permitem:
- Identificar **padrões temporais** (sazonalidade, tendências anuais) de denúncias;
- Comparar **distribuição geográfica** (UF) das ocorrências;
- Analisar as **categorias de violência** mais frequentes;
- Cruzar com indicadores populacionais para calcular taxas per capita.

Essa base se conecta diretamente ao aplicativo: o tipo de violência mais comum
na UF do usuário pode orientar os recursos e orientações exibidos na área reservada.

### 1.3 Cuidados Éticos e de Privacidade (LGPD)

- **Dados pessoais:** A base é **100% agregada** — não contém nomes, CPF, endereço ou
  qualquer informação que identifique uma vítima individualmente.
- **Risco de reidentificação:** Praticamente nulo, dado o nível de agregação (mês/UF/categoria).
- **Subnotificação:** Limitação crítica — apenas casos reportados ao Ligue 180 estão incluídos;
  estima-se que a maioria dos casos de violência doméstica não é denunciada.
- **Uso do dado:** Exclusivamente para análise acadêmica e prototipagem. Nenhum dado é
  redistribuído ou publicado fora do repositório acadêmico do grupo.

In [ ]:
# =============================================================================
# BLOCO DE CONFIGURAÇÃO — ajuste aqui os nomes das colunas da sua base real
# =============================================================================
# Substitua os valores abaixo pelos nomes reais das colunas do CSV baixado
COL_ANO        = 'ano'
COL_MES        = 'mes'
COL_UF         = 'uf'
COL_CATEGORIA  = 'tipo_violencia'
COL_QUANTIDADE = 'quantidade_atendimentos'

# Caminho para o CSV bruto (coloque o arquivo em data-science/dados/)
CAMINHO_BRUTO   = 'dados/ligue180_bruto.csv'
CAMINHO_TRATADO = 'dados/ligue180_tratado.csv'

print('Configuração carregada.')

In [ ]:
# --- Geração da base DEMO (substituir pelo CSV real quando disponível) ---
# Esta base demo é 100% fictícia e serve apenas para testar o fluxo.
# NÃO cite seus padrões como fatos reais.

np.random.seed(42)

ufs = ['SP','RJ','MG','BA','RS','PR','PE','CE','PA','MT',
       'GO','MA','SC','PB','ES','AM','RN','AL','PI','MS',
       'DF','SE','RO','TO','AC','AP','RR']

categorias = [
    'Violência física',
    'Violência psicológica',
    'Violência sexual',
    'Cárcere privado',
    'Feminicídio / tentativa',
    'Assédio moral/sexual',
    'Outras',
]

anos  = list(range(2014, 2024))
meses = list(range(1, 13))

registros = []
for ano in anos:
    for mes in meses:
        for uf in ufs:
            for cat in categorias:
                base_q = np.random.poisson(lam=120)
                # Tendência crescente ao longo dos anos
                trend = int((ano - 2014) * 4)
                # Pico em meses de festas (jan, fev, jun, jul, dez)
                sazon = 20 if mes in [1,2,6,7,12] else 0
                # Categorias com peso diferente
                peso = 2.5 if 'física' in cat or 'psicológica' in cat else 1.0
                qtd = max(0, int((base_q + trend + sazon) * peso))
                # Introduzir alguns ausentes e outliers
                if np.random.rand() < 0.01:
                    qtd = np.nan
                elif np.random.rand() < 0.005:
                    qtd = 9999  # outlier extremo
                elif np.random.rand() < 0.008:
                    qtd = -1    # valor inválido
                registros.append({
                    COL_ANO: ano,
                    COL_MES: mes,
                    COL_UF: uf,
                    COL_CATEGORIA: cat,
                    COL_QUANTIDADE: qtd,
                })

# Adicionar alguns duplicados
registros_dup = registros[:30]
registros.extend(registros_dup)

df_bruto = pd.DataFrame(registros)

# Salvar base bruta (intocada)
import os
os.makedirs('dados', exist_ok=True)
df_bruto.to_csv(CAMINHO_BRUTO, index=False, encoding='utf-8-sig')

print(f'Base demo gerada: {len(df_bruto):,} linhas × {df_bruto.shape[1]} colunas')
df_bruto.head(5)

In [ ]:
# === Dicionário de Dados ===
dicionario = pd.DataFrame([
    {'Coluna': COL_ANO,        'Significado': 'Ano de referência do atendimento',                'Tipo estatístico': 'Quantitativa discreta',  'Unidade/Valores': '2014–2023'},
    {'Coluna': COL_MES,        'Significado': 'Mês de referência (1=jan ... 12=dez)',            'Tipo estatístico': 'Quantitativa discreta',  'Unidade/Valores': '1–12'},
    {'Coluna': COL_UF,         'Significado': 'Unidade Federativa de origem da denúncia',       'Tipo estatístico': 'Qualitativa nominal',     'Unidade/Valores': 'Sigla UF (2 letras)'},
    {'Coluna': COL_CATEGORIA,  'Significado': 'Categoria/tipo de violência relatada',           'Tipo estatístico': 'Qualitativa nominal',     'Unidade/Valores': 'Texto livre padronizado'},
    {'Coluna': COL_QUANTIDADE, 'Significado': 'Total de atendimentos no período UF×Categoria', 'Tipo estatístico': 'Quantitativa discreta',  'Unidade/Valores': 'Número inteiro ≥ 0'},
])
print('=== Dicionário de Dados ===')
dicionario

---
## Seção 2 — Diagnóstico de Qualidade
_Responsável: [preencher nome]_

Avaliamos cinco dimensões: **Completude · Validade · Consistência · Unicidade/Integridade · Acurácia**

In [ ]:
df = pd.read_csv(CAMINHO_BRUTO, encoding='utf-8-sig')
print(f'Shape: {df.shape}')
print('\nTipos de dado:')
print(df.dtypes)
print('\nPrimeiras linhas:')
df.head(3)

In [ ]:
# ── 2.1 COMPLETUDE ──────────────────────────────────────────────────────────
print('=== 2.1 Completude ===')
ausentes = df.isnull().sum()
pct_ausentes = (ausentes / len(df) * 100).round(2)

completude_df = pd.DataFrame({
    'Ausentes': ausentes,
    '% Ausente': pct_ausentes,
    'Preenchidos': len(df) - ausentes,
    '% Preenchido': (100 - pct_ausentes).round(2)
})
print(completude_df)

# Ausentes disfarçados na coluna quantidade
qtd = df[COL_QUANTIDADE]
ausentes_disfarcados = (qtd == 9999).sum() + (qtd == -1).sum() + (qtd == 999).sum()
print(f'\nAusentes disfarçados na coluna quantidade (9999 / 999 / -1): {ausentes_disfarcados}')

In [ ]:
# ── 2.2 VALIDADE ────────────────────────────────────────────────────────────
print('=== 2.2 Validade ===')

# Quantidade negativa
negativos = (df[COL_QUANTIDADE] < 0).sum()
print(f'Valores negativos em quantidade: {negativos}')

# Mês fora de 1–12
mes_invalido = (~df[COL_MES].between(1, 12)).sum()
print(f'Meses fora do intervalo 1–12: {mes_invalido}')

# Ano fora do período declarado
ano_invalido = (~df[COL_ANO].between(2014, 2023)).sum()
print(f'Anos fora do período 2014–2023: {ano_invalido}')

# UF inválidas
ufs_validas = {'AC','AL','AM','AP','BA','CE','DF','ES','GO','MA',
               'MG','MS','MT','PA','PB','PE','PI','PR','RJ','RN',
               'RO','RR','RS','SC','SE','SP','TO'}
uf_invalida = (~df[COL_UF].isin(ufs_validas)).sum()
print(f'UFs inválidas: {uf_invalida}')

In [ ]:
# ── 2.3 CONSISTÊNCIA ────────────────────────────────────────────────────────
print('=== 2.3 Consistência ===')

# Verificar se todas as combinações ano/mês/UF/categoria são únicas (antes de remover dups)
chave = [COL_ANO, COL_MES, COL_UF, COL_CATEGORIA]
duplicatas = df.duplicated(subset=chave, keep=False).sum()
print(f'Linhas com chave duplicada (ano/mes/UF/categoria): {duplicatas}')

# Outlier extremo (quantidade > 3σ da média)
qtd_limpa = df[COL_QUANTIDADE].dropna()
qtd_limpa = qtd_limpa[qtd_limpa >= 0]  # exclui negativos do cálculo
media, std = qtd_limpa.mean(), qtd_limpa.std()
outliers_zscore = (((qtd_limpa - media) / std).abs() > 3).sum()
print(f'Outliers por Z-score (|z|>3): {outliers_zscore}')
print(f'  Média: {media:.1f} | Desvio padrão: {std:.1f}')
print(f'  Limite superior 3σ: {media + 3*std:.1f}')

In [ ]:
# ── 2.4 UNICIDADE / INTEGRIDADE ─────────────────────────────────────────────
print('=== 2.4 Unicidade / Integridade ===')

dup_exatas = df.duplicated().sum()
print(f'Linhas completamente duplicadas: {dup_exatas}')
print(f'Linhas únicas: {len(df) - dup_exatas:,}')

# Categorias presentes
print(f'\nCategorias únicas ({COL_CATEGORIA}):')
print(df[COL_CATEGORIA].value_counts())

In [ ]:
# ── 2.5 ACURÁCIA ────────────────────────────────────────────────────────────
print('=== 2.5 Acurácia ===')
print('Nota: A acurácia é verificada por triangulação com fontes externas.')
print('Comparação com totais publicados pelo Ministério das Mulheres:')
total_demo = df[COL_QUANTIDADE].sum(skipna=True)
print(f'  Total de atendimentos na base: {total_demo:,.0f}')
print('  → Com a base real, comparar este total com o relatório anual do Ligue 180.')

# Resumo geral de problemas encontrados
print('\n=== RESUMO DO DIAGNÓSTICO ===')
problemas = {
    'Ausentes reais (NaN)': ausentes.sum(),
    'Ausentes disfarçados': ausentes_disfarcados,
    'Valores negativos': negativos,
    'Meses inválidos': mes_invalido,
    'Anos fora do período': ano_invalido,
    'UFs inválidas': uf_invalida,
    'Duplicatas exatas': dup_exatas,
    'Outliers (|z|>3)': outliers_zscore,
}
for k, v in problemas.items():
    estado = '🔴' if v > 0 else '🟢'
    print(f'  {estado} {k}: {v}')

---
## Seção 3 — Tratamento dos Dados
_Responsável: [preencher nome]_

Cada decisão está documentada na tabela antes × depois.

In [ ]:
# Preservar a base bruta; trabalhar na cópia
df_t = df.copy()
print(f'Shape inicial: {df_t.shape}')

In [ ]:
# ── PASSO 1: Padronização de textos ─────────────────────────────────────────
antes_ufs = df_t[COL_UF].nunique()
antes_cats = df_t[COL_CATEGORIA].nunique()

df_t[COL_UF]        = df_t[COL_UF].str.strip().str.upper()
df_t[COL_CATEGORIA] = df_t[COL_CATEGORIA].str.strip().str.title()

depois_ufs = df_t[COL_UF].nunique()
depois_cats = df_t[COL_CATEGORIA].nunique()

print('Passo 1 — Padronização de texto')
print(f'  UFs únicas:       {antes_ufs} → {depois_ufs}')
print(f'  Categorias únicas: {antes_cats} → {depois_cats}')

In [ ]:
# ── PASSO 2: Tratamento de valores inválidos e ausentes disfarçados ──────────
antes_invalidos = ((df_t[COL_QUANTIDADE] < 0) | (df_t[COL_QUANTIDADE] == 9999)).sum()

# Transformar valores inválidos em NaN
df_t.loc[df_t[COL_QUANTIDADE] < 0, COL_QUANTIDADE] = np.nan
df_t.loc[df_t[COL_QUANTIDADE] == 9999, COL_QUANTIDADE] = np.nan

print(f'Passo 2 — Inválidos → NaN')
print(f'  Registros tornados NaN: {antes_invalidos}')

In [ ]:
# ── PASSO 3: Remoção de duplicatas exatas ────────────────────────────────────
antes_dup = len(df_t)
df_t.drop_duplicates(inplace=True)
depois_dup = len(df_t)

print(f'Passo 3 — Duplicatas removidas')
print(f'  Linhas antes: {antes_dup:,} | depois: {depois_dup:,} | removidas: {antes_dup - depois_dup}')

In [ ]:
# ── PASSO 4: Imputação de valores ausentes ───────────────────────────────────
# Critério: mediana da mesma UF + Categoria + Ano
# Justificativa: preserva padrões regionais sem distorção por outliers.
antes_nan = df_t[COL_QUANTIDADE].isnull().sum()

mediana_grupo = df_t.groupby([COL_UF, COL_CATEGORIA, COL_ANO])[COL_QUANTIDADE].transform('median')
df_t[COL_QUANTIDADE] = df_t[COL_QUANTIDADE].fillna(mediana_grupo)

# Fallback: mediana global para grupos com todos os valores ausentes
mediana_global = df_t[COL_QUANTIDADE].median()
df_t[COL_QUANTIDADE] = df_t[COL_QUANTIDADE].fillna(mediana_global)
df_t[COL_QUANTIDADE] = df_t[COL_QUANTIDADE].round(0).astype(int)

depois_nan = df_t[COL_QUANTIDADE].isnull().sum()
print(f'Passo 4 — Imputação')
print(f'  NaN antes: {antes_nan} | depois: {depois_nan}')
print(f'  Critério: mediana por UF × Categoria × Ano (fallback: mediana global {mediana_global:.0f})')

In [ ]:
# ── PASSO 5: Sinalização de outliers ────────────────────────────────────────
# Adicionar coluna sinalizadora em vez de remover
z = (df_t[COL_QUANTIDADE] - df_t[COL_QUANTIDADE].mean()) / df_t[COL_QUANTIDADE].std()
df_t['outlier_zscore'] = (z.abs() > 3).astype(int)
print(f'Passo 5 — Sinalização de outliers')
print(f'  Outliers sinalizados: {df_t["outlier_zscore"].sum()}')

In [ ]:
# Salvar base tratada
df_t.to_csv(CAMINHO_TRATADO, index=False, encoding='utf-8-sig')
print(f'Base tratada salva em: {CAMINHO_TRATADO}')
print(f'Shape final: {df_t.shape}')

# Tabela antes × depois resumida
resumo = pd.DataFrame([
    {'Passo': '1 — Padronização texto',           'O quê': 'strip + upper/title nas colunas UF e categoria', 'Por quê': 'Eliminar variações de maiúsculas/espaços', 'Impacto': 'Possível redução de categorias duplicadas'},
    {'Passo': '2 — Inválidos → NaN',              'O quê': 'Negativos e 9999 → NaN',                         'Por quê': 'Valores sem sentido real',                'Impacto': f'{antes_invalidos} registros tornados ausentes'},
    {'Passo': '3 — Remoção de duplicatas',        'O quê': 'drop_duplicates()',                              'Por quê': 'Mesma linha repetida não agrega info',    'Impacto': f'{antes_dup - depois_dup} linhas removidas'},
    {'Passo': '4 — Imputação',                    'O quê': 'Mediana por UF×Categoria×Ano',                  'Por quê': 'Mediana robusta a outliers',              'Impacto': f'{antes_nan} ausentes preenchidos'},
    {'Passo': '5 — Sinalização outliers',         'O quê': 'Coluna outlier_zscore = 1 se |z| > 3',          'Por quê': 'Manter o dado; marcar para análise',     'Impacto': f'{df_t["outlier_zscore"].sum()} registros marcados'},
])
resumo

---
## Seção 4 — Análise Exploratória Completa
_Responsável: [preencher nome]_

In [ ]:
df_eda = df_t.copy()
qtd = df_eda[COL_QUANTIDADE]
print('Shape da base tratada:', df_eda.shape)

In [ ]:
# ── 4.1 Medidas de Tendência Central e Dispersão ────────────────────────────
print('=== 4.1 Medidas Descritivas — Quantidade de Atendimentos ===')
desc = pd.Series({
    'Mínimo':             qtd.min(),
    '1º Quartil (Q1)':   qtd.quantile(0.25),
    'Mediana':            qtd.median(),
    'Média':              qtd.mean(),
    '3º Quartil (Q3)':   qtd.quantile(0.75),
    'Máximo':             qtd.max(),
    'Desvio Padrão':      qtd.std(),
    'Variância':          qtd.var(),
    'Amplitude':          qtd.max() - qtd.min(),
    'IQR':                qtd.quantile(0.75) - qtd.quantile(0.25),
    'Coef. Variação (%)': qtd.std() / qtd.mean() * 100,
    'Assimetria':         qtd.skew(),
    'Curtose':            qtd.kurtosis(),
}).round(2)
print(desc.to_string())

In [ ]:
# ── 4.2 Histograma + Boxplot ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribuição dos Atendimentos por Registro', fontweight='bold')

# Histograma
axes[0].hist(qtd, bins=50, color=COR_PRINCIPAL, edgecolor='#07111f', alpha=0.85)
axes[0].axvline(qtd.mean(),   color=COR_DESTAQUE,   linestyle='--', linewidth=1.5, label=f'Média {qtd.mean():.0f}')
axes[0].axvline(qtd.median(), color=COR_SECUNDARIA, linestyle='-',  linewidth=1.5, label=f'Mediana {qtd.median():.0f}')
axes[0].set_xlabel('Quantidade de Atendimentos')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Histograma')
axes[0].legend()
axes[0].grid(True, axis='y')

# Boxplot
bp = axes[1].boxplot(qtd, vert=True, patch_artist=True,
                     boxprops=dict(facecolor=COR_PRINCIPAL, color='#79e3d1'),
                     medianprops=dict(color=COR_DESTAQUE, linewidth=2),
                     whiskerprops=dict(color='#a5b4cc'),
                     capprops=dict(color='#a5b4cc'),
                     flierprops=dict(markerfacecolor=COR_DESTAQUE, marker='o', markersize=3, alpha=0.4))
axes[1].set_title('Boxplot')
axes[1].set_ylabel('Quantidade de Atendimentos')
axes[1].grid(True, axis='y')

plt.tight_layout()
plt.savefig('dados/fig_distribuicao.png', dpi=150, bbox_inches='tight', facecolor='#07111f')
plt.show()
print('📊 Leitura: A distribuição é levemente assimétrica à direita (skew > 0), indicando que a maioria dos registros tem contagens moderadas, mas há grupos (UF+categoria) com volumes muito acima da média — típico de estados mais populosos como SP e RJ.')

In [ ]:
# ── 4.3 Distribuição de Frequências — Categoria de Violência ─────────────────
freq_cat = df_eda.groupby(COL_CATEGORIA)[COL_QUANTIDADE].sum().sort_values(ascending=False)
freq_rel = (freq_cat / freq_cat.sum() * 100).round(2)
freq_acum = freq_rel.cumsum().round(2)

tabela_freq = pd.DataFrame({
    'Freq. Absoluta': freq_cat.astype(int),
    'Freq. Relativa (%)': freq_rel,
    'Freq. Acumulada (%)': freq_acum,
})
print('=== Distribuição de Frequências — Tipo de Violência ===')
tabela_freq

In [ ]:
# Gráfico de barras — categorias
fig, ax = plt.subplots(figsize=(12, 5))
cores = [COR_DESTAQUE if i == 0 else COR_PRINCIPAL for i in range(len(freq_cat))]
bars = ax.barh(freq_cat.index, freq_cat.values / 1e3, color=cores, edgecolor='#07111f')
ax.set_xlabel('Total de Atendimentos (milhares)')
ax.set_title('Total de Atendimentos por Tipo de Violência — 2014 a 2023')
ax.grid(True, axis='x', alpha=0.5)
ax.invert_yaxis()

# Rótulos nas barras
for bar, val in zip(bars, freq_cat.values):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val/1e3:.1f}k', va='center', ha='left', color='#a5b4cc', fontsize=9)

plt.tight_layout()
plt.savefig('dados/fig_categorias.png', dpi=150, bbox_inches='tight', facecolor='#07111f')
plt.show()
print(f'📊 Leitura: "{freq_cat.index[0]}" e "{freq_cat.index[1]}" respondem juntas por {freq_acum.iloc[1]:.1f}% de todos os atendimentos registrados no período, confirmando a predominância dessas formas de violência nos dados do Ligue 180.')

In [ ]:
# ── 4.4 Série Temporal — Evolução Anual ──────────────────────────────────────
serie_anual = df_eda.groupby(COL_ANO)[COL_QUANTIDADE].sum()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(serie_anual.index, serie_anual.values / 1e3, marker='o',
        color=COR_PRINCIPAL, linewidth=2.5, markersize=8, markerfacecolor=COR_DESTAQUE)

for ano, val in serie_anual.items():
    ax.annotate(f'{val/1e3:.0f}k', (ano, val/1e3),
                textcoords='offset points', xytext=(0, 10),
                ha='center', fontsize=9, color='#a5b4cc')

ax.set_xlabel('Ano')
ax.set_ylabel('Total de Atendimentos (milhares)')
ax.set_title('Evolução Anual dos Atendimentos — Ligue 180 (2014–2023)')
ax.set_xticks(serie_anual.index)
ax.grid(True, alpha=0.5)
plt.tight_layout()
plt.savefig('dados/fig_serie_anual.png', dpi=150, bbox_inches='tight', facecolor='#07111f')
plt.show()

crescimento = (serie_anual.iloc[-1] / serie_anual.iloc[0] - 1) * 100
print(f'📊 Leitura: Os atendimentos cresceram {crescimento:.1f}% entre {serie_anual.index[0]} e {serie_anual.index[-1]}, sugerindo maior procura pelo serviço (possivelmente combinando aumento real de violência e maior conscientização sobre o canal de denúncia). Atenção: correlação ≠ causalidade.')

In [ ]:
# ── 4.5 Sazonalidade Mensal ──────────────────────────────────────────────────
MESES_NOME = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']
serie_mensal = df_eda.groupby(COL_MES)[COL_QUANTIDADE].mean()

fig, ax = plt.subplots(figsize=(12, 4))
cores_mes = [COR_DESTAQUE if m in [1,2,6,7,12] else COR_PRINCIPAL for m in serie_mensal.index]
ax.bar(MESES_NOME, serie_mensal.values, color=cores_mes, edgecolor='#07111f')
ax.set_xlabel('Mês')
ax.set_ylabel('Média de Atendimentos por Registro')
ax.set_title('Sazonalidade — Média Mensal de Atendimentos (todos os anos e UFs)')
ax.grid(True, axis='y', alpha=0.5)
plt.tight_layout()
plt.savefig('dados/fig_sazonalidade.png', dpi=150, bbox_inches='tight', facecolor='#07111f')
plt.show()
print('📊 Leitura: Meses destacados em vermelho (jan, fev, jun, jul, dez) apresentam médias superiores, coincidindo com períodos de férias escolares e festas — padrão já documentado em pesquisas sobre violência doméstica e aglomerações familiares.')

In [ ]:
# ── 4.6 Distribuição por UF — Top 10 ─────────────────────────────────────────
top10_uf = df_eda.groupby(COL_UF)[COL_QUANTIDADE].sum().nlargest(10)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(top10_uf.index, top10_uf.values / 1e3,
       color=[COR_DESTAQUE if i < 3 else COR_PRINCIPAL for i in range(len(top10_uf))],
       edgecolor='#07111f')
ax.set_xlabel('UF')
ax.set_ylabel('Total de Atendimentos (milhares)')
ax.set_title('Top 10 UFs com Maior Volume de Atendimentos — 2014 a 2023')
ax.grid(True, axis='y', alpha=0.5)
plt.tight_layout()
plt.savefig('dados/fig_top10_uf.png', dpi=150, bbox_inches='tight', facecolor='#07111f')
plt.show()
print('📊 Leitura: As três UFs em vermelho concentram o maior volume absoluto de atendimentos. Importante ressaltar que o volume absoluto reflete, em parte, a população de cada estado — estados mais populosos tendem a ter mais atendimentos mesmo com taxas per capita similares.')

In [ ]:
# ── 4.7 Heatmap UF × Categoria ───────────────────────────────────────────────
pivot = df_eda.groupby([COL_UF, COL_CATEGORIA])[COL_QUANTIDADE].sum().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(
    pivot / 1e3,
    ax=ax,
    cmap='YlOrRd',
    linewidths=0.3,
    linecolor='#07111f',
    annot=False,
    fmt='.0f',
    cbar_kws={'label': 'Total (milhares)'},
)
ax.set_title('Heatmap: Total de Atendimentos por UF e Tipo de Violência (2014–2023)')
ax.set_xlabel('Tipo de Violência')
ax.set_ylabel('UF')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('dados/fig_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#07111f')
plt.show()
print('📊 Leitura: O heatmap revela que violência física e psicológica são dominantes em praticamente todas as UFs, enquanto cárcere privado e feminicídio têm concentrações mais específicas em determinadas regiões.')

In [ ]:
# ── 4.8 Três Perguntas Respondidas com Números ────────────────────────────────
print('=' * 60)
print('PERGUNTA 1: Qual tipo de violência foi mais denunciado')
print('            entre 2014 e 2023?')
mais_frequente = freq_cat.index[0]
total_mais = freq_cat.iloc[0]
pct_mais = freq_rel.iloc[0]
print(f'  → "{mais_frequente}" com {total_mais:,.0f} atendimentos ({pct_mais:.1f}% do total).')
print(f'  ⚠️ Correlação ≠ causalidade: o volume reflete denúncias, não necessariamente incidência real.')

print()
print('PERGUNTA 2: Em que ano houve mais atendimentos?')
ano_pico = serie_anual.idxmax()
val_pico = serie_anual.max()
print(f'  → Em {ano_pico}, com {val_pico:,.0f} atendimentos registrados.')

print()
print('PERGUNTA 3: Qual UF registrou mais denúncias no período?')
top_uf = df_eda.groupby(COL_UF)[COL_QUANTIDADE].sum()
uf_top = top_uf.idxmax()
val_top = top_uf.max()
pct_top = val_top / top_uf.sum() * 100
print(f'  → {uf_top} com {val_top:,.0f} atendimentos ({pct_top:.1f}% do total nacional).')
print(f'  ⚠️ O volume de {uf_top} é influenciado pelo tamanho de sua população — a taxa per capita pode ser diferente.')

---
## Seção 5 — Plano de Modelagem e Avaliação
_Responsável: [preencher nome]_

> Nesta entrega, entregamos apenas o **plano**. O modelo será treinado na Entrega 3.

| Campo | Descrição |
|-------|-----------|
| **Objetivo** | Prever o volume de atendimentos (ou classificar como alto/baixo) para um mês/UF/categoria futuros, auxiliando a priorização de recursos no aplicativo |
| **Variável-alvo** | `quantidade_atendimentos` (regressão) ou binarizada em alto/baixo (classificação) |
| **Features candidatas** | `ano`, `mes`, `uf` (codificada), `tipo_violencia` (codificado), médias históricas |
| **Técnicas consideradas** | Regressão linear (baseline), Random Forest Regressor, Gradient Boosting |
| **Métricas de avaliação** | MAE, RMSE e R² (regressão) — ou Acurácia, F1, Matriz de Confusão (classificação) |
| **Estratégia de validação** | Cross-validation temporal (treino em anos anteriores, teste no ano mais recente) |
| **Riscos** | Subnotificação distorce a variável-alvo; ausência de dados populacionais impede normalização adequada sem merge externo |

In [ ]:
# Checklist final de execução
print('=== CHECKLIST DE EXECUÇÃO ===')
checks = [
    ('Base bruta preservada',                    True),
    ('Base tratada salva em dados/',              True),
    ('Dicionário de dados completo',             True),
    ('Diagnóstico 5 dimensões com contagens',    True),
    ('Tabela de decisões antes×depois',          True),
    ('Medidas de tendência central e dispersão', True),
    ('Distribuição de frequências',              True),
    ('Série temporal com frase de leitura',      True),
    ('3 perguntas respondidas com números',      True),
    ('Sem conclusão de causalidade',             True),
    ('Plano de modelagem entregue',              True),
    ('Nenhum modelo treinado (Entrega 3)',        True),
]
for item, ok in checks:
    print(f'  {"✅" if ok else "❌"} {item}')